# Characteristic Portfolio Formation

Two investors can use the same characteristic and still hold different investments. One may compare firms with the entire market, another with firms of similar size. They may update holdings at different dates or allocate capital differently across the same securities. These choices determine the portfolio return being studied.

Portfolio formation connects the measurements in [01](01_firm_characteristics.ipynb) to actual holdings. We first explain how breakpoints, sorting procedures, formation dates and weights define an investment. We then apply that framework to observed securities in the prepared data, comparing their June assignments before constructing historical characteristic-portfolio returns.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name=='notebooks' else ROOT
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from factor_asset_pricing.characteristics import fama_french_characteristics
from factor_asset_pricing.sorts import quantile_breakpoints, assign_portfolios, assign_annual_portfolios, portfolio_returns
NAVY, BLUE, TEAL, AMBER, GRAY = '#17324D', '#2878A5', '#168A86', '#D99A32', '#737D87'
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':12,
    'axes.labelsize':11,'xtick.labelsize':10,'ytick.labelsize':10,'legend.fontsize':10,
    'figure.dpi':110,'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,'lines.linewidth':1.8,'grid.alpha':.15})

from factor_asset_pricing.wrds_data import load_prepared_data
RAW_WRDS = ROOT / "data/raw/wrds"
PROCESSED = ROOT / "data/processed"


## 1. Characteristic Portfolio Formation

### 1.1 Why form characteristic portfolios?

Characteristic portfolios make an investment comparison explicit. A size sort compares securities with different market capitalizations, while a valuation sort compares securities with different valuation measurements. We can subsequently examine whether their returns, exposures or abnormal means differ. The sort defines the comparison, while the regressions in 04 evaluate it.

Grouping firms also diversifies some firm-specific return noise. A single firm’s outcome can be dominated by events unique to its business. Combining holdings makes shared patterns easier to study, although diversification does not eliminate all residual risk or establish abnormal return. The portfolios must remain sufficiently distinct to answer the investment question and sufficiently populated to avoid relying on a few idiosyncratic outcomes.

### 1.2 Investment universes and breakpoints

The investment universe contains securities eligible for the holdings. The reference universe supplies the distribution used to choose boundaries. The two need not coincide. A selected exchange can define cutoffs applied to securities on several exchanges. Changing the reference sample changes the comparison even when the eligible holdings remain fixed.

Breakpoints divide the reference characteristic distribution into groups. Quintiles use the 20th, 40th, 60th and 80th percentiles. Firms receive labels according to where their measurements fall relative to those values. A quintile label describes the boundary rule, rather than guaranteeing equally populated investment portfolios.

Ties, missing signals and a broader investment universe can all change cell counts. The implementation must state the quantile convention and which side receives a value exactly at a boundary. Missing characteristics or unavailable reference cutoffs leave a security unassigned, rather than placing it arbitrarily in an extreme group.

### 1.3 Univariate, independent and conditional sorts

A **univariate sort** divides firms by one characteristic. Its advantage is a direct link between the measurement and portfolio labels. Other attributes can nevertheless differ across the groups. A valuation comparison may also compare different size compositions.

An **independent sort** calculates each characteristic’s breakpoints separately, then intersects the assignments. In a size × valuation grid, the same valuation boundaries apply across all size groups. The cells may be unevenly populated when the characteristics are related. Independent sorting refers to separate cutoff calculations, not statistical independence of firms’ characteristics or returns.

A **conditional sort** first assigns firms on one characteristic, then recalculates the next characteristic’s boundaries within each earlier group. A high-valuation small firm is now high relative to the specified reference firms in its size category. It need not exceed the high-valuation threshold among large firms. The sorting order therefore changes the investment comparison.

Conditioning can make category comparisons more balanced, but it does not automatically remove continuous characteristic differences, hedge return exposures or identify a causal effect. Additional splits also reduce the firms available to each portfolio, potentially increasing concentration and noise.

### 1.4 Formation and rebalancing

Formation assigns eligible securities to portfolios using information available at the decision date. A June decision can establish membership for July through the following June. The next formation date can move a security into a different group. The June return still belongs to the preceding assignment, while the new assignment starts in July.

Rebalancing concerns the positions and capital weights maintained within those groups. Membership and weights need not stay fixed together. A firm can remain in the same annual group while its share of portfolio market value changes each month. Annual formation therefore does not imply twelve months of constant security weights.

Monthly formation updates labels more frequently and can respond faster to changing signals, with different turnover implications. Both annual and monthly designs can produce monthly return series. The researcher must specify the membership rule, weight-updating rule and treatment of entrants and exits. Measurement availability was established in 01 and remains a requirement here.

### 1.5 Portfolio weights and returns

Once membership and holding dates are established, weighting determines how much capital each security receives. For portfolio $p$ and month $t$,

$$R_{p,t}=\sum_{i\in\mathcal P_{p,t}}w_{i,t-1}R_{i,t},\qquad\sum_iw_{i,t-1}=1.$$

$\mathcal P_{p,t}$ identifies the holdings and $w_{i,t-1}$ their pre-return capital shares. A 20% allocation to a security earning 5% contributes one percentage point to the portfolio return. The weights are chosen before that payoff is observed.

Equal weights give each included security the same capital share, while value weights use beginning-period market equity:

$$w^{EW}_{i,t-1}=\frac1{n_p},\qquad
w^{VW}_{i,t-1}=\frac{ME_{i,t-1}}{\sum_{j\in p}ME_{j,t-1}}.$$

Compared with value weighting, equal weighting gives smaller securities more relative representation. Changing the weights can change the return even when the holdings are identical. Missing realized returns require an explicit accounting rule. Dropping them and renormalizing observed weights describes an available-return sample, rather than a complete ex-ante policy for an implementable portfolio.

For portfolio $p$, the excess return is

$$\text{Portfolio excess return}=R_{p,t}-R_{f,t}.$$

where $R_{f,t}$ is the risk-free return for the same month.

Dates, frequency and units must agree. Subtract the risk-free return once from a fully invested portfolio. A long–short difference between two such portfolios already cancels the common risk-free component, as developed in 03.


## 2. Example: Fama–French Characteristic Portfolios

The Fama–French portfolio designs provide a familiar application of this framework. I first form size × book-to-market portfolios using the June-2020 measurements, then compare independent and conditional assignments. Locally constructed historical size × BM, profitability and investment portfolios provide the broader return comparison at the end.

### 2.1 Data and formation conventions

The June illustration uses the same linked firm panel and measurement conventions as 01. The starting universe contains positive-capitalization NYSE, AMEX and Nasdaq securities. For the size × BM comparison, both characteristics must be usable. Eligible NYSE securities define the quintile boundaries, which are applied to all eligible securities.

June size, prior-year book equity and December market equity determine the assignments. Independent annual inputs from 00 supply fiscal-2019 accounts without reading a future July carrier. Their July holding availability is assumed, not a verified publication date. Market equity remains security-level under the existing public API.

The June example lets us see which securities enter each group. I then use the same locally prepared observations to construct historical portfolio returns. It does not certify exact official French eligibility. Accounting conventions and provenance limitations remain those explained in 01.


In [ ]:
FORMATION_YEAR = 2020
formation_month = pd.Timestamp(FORMATION_YEAR, 6, 1)
formation_date = pd.Timestamp(FORMATION_YEAR, 6, 30)
december_month = pd.Timestamp(FORMATION_YEAR-1, 12, 1)
raw, original_annual = load_prepared_data(PROCESSED, RAW_WRDS,
    start=f"{FORMATION_YEAR-1}-01-01", end=formation_month)
assert not raw.duplicated(["permno", "mdate"]).any()
# Independent annual inputs, not accounts recovered from a future July row.
annual = original_annual.loc[
    original_annual.fyear.isin([FORMATION_YEAR-2, FORMATION_YEAR-1]) &
    original_annual.available_from.le(pd.Timestamp(FORMATION_YEAR, 7, 1))
].copy()
assert not annual.duplicated(["gvkey", "fyear"]).any()
api_panel = raw.rename(columns={"date": "crsp_date", "mdate": "date"})
measured = fama_french_characteristics(api_panel, annual_data=annual)
formation = measured.loc[measured.date.eq(formation_month) & measured.exchcd.isin([1,2,3]) &
                          measured.me.gt(0), ['permno','gvkey','date','exchcd','me','me_dec','bm','op','inv']].copy()
formation['size_millions'] = formation.me/1000
formation = formation.sort_values('permno').reset_index(drop=True)
assert len(formation)>0 and formation.permno.is_unique
measure_columns = ['size_millions','bm','op','inv']
print(f'June {FORMATION_YEAR}: {len(formation):,} eligible security records in the locally prepared linked universe.')
print('Independent annual records use the assumed July holding-availability convention, not verified filing dates.')


### 2.2 From characteristics to portfolio assignments

An independent $5\times5$ size × BM design uses four common boundaries for each characteristic. The library chooses cutoffs from the observed reference values. When a percentile falls between two ordered observations, it uses the lower value rather than averaging them. A security exactly on a cutoff enters the lower group. This convention makes assignments reproducible, although other implementations can choose slightly different boundaries.

The table shows the NYSE boundaries used for the assignments. To see what changes when the comparison group changes, I compare independent sorts with size-conditional BM sorts. Both use the same eligible securities and NYSE reference rule. The conditional design recalculates BM boundaries among reference securities inside each size group. It is a comparison design, rather than the independent Fama–French construction.

In [ ]:
eligible = formation.loc[formation.bm.notna()].copy()
chars = ['size_millions','bm']
labels = ['size_millions_portfolio','bm_portfolio']
reference = eligible.loc[eligible.exchcd.eq(1)]
assert len(reference)>0
probabilities = np.arange(1,5)/5
cuts = {name:quantile_breakpoints(reference[name],5) for name in chars}
breakpoint_table = pd.DataFrame({'Percentile (%)':100*probabilities,
    'June size ($m)':cuts['size_millions'],'Book-to-market (ratio)':cuts['bm']}).set_index('Percentile (%)')
print(f'{len(eligible):,} eligible June securities, including {len(reference):,} NYSE reference securities.')
display(breakpoint_table.round(6))
independent = assign_portfolios(eligible,chars,5,reference='nyse',exchange_col='exchcd')
conditional = assign_portfolios(eligible,chars,5,reference='nyse',exchange_col='exchcd',
                               conditional_on={'bm':['size_millions']})
# Independently select order statistics and assign boundaries with searchsorted.
for name in chars:
    values = np.sort(reference[name].to_numpy(dtype=float))
    hand_cuts = values[np.floor((len(values)-1)*probabilities).astype(int)]
    np.testing.assert_allclose(cuts[name],hand_cuts)
    expected = np.searchsorted(hand_cuts,eligible[name].to_numpy(dtype=float),side='left')+1
    np.testing.assert_array_equal(independent[name+'_portfolio'].to_numpy(dtype=int),expected)
for size_group in range(1,6):
    group = conditional.loc[conditional.size_millions_portfolio.eq(size_group)]
    local_reference = group.loc[group.exchcd.eq(1),'bm'].to_numpy(dtype=float)
    assert len(local_reference)>0
    values = np.sort(local_reference)
    boundaries = values[np.floor((len(values)-1)*probabilities).astype(int)]
    expected = np.searchsorted(boundaries,group.bm.to_numpy(dtype=float),side='left')+1
    np.testing.assert_array_equal(group.bm_portfolio.to_numpy(dtype=int),expected)
counts = {name:assigned.groupby(labels,observed=True).size().unstack(fill_value=0)
          .reindex(index=range(1,6),columns=range(1,6),fill_value=0)
          for name,assigned in [('Independent',independent),('Conditional',conditional)]}
assert all(int(table.to_numpy().sum())==len(eligible) for table in counts.values())
changed = int((independent.bm_portfolio != conditional.bm_portfolio).sum())
print(f'{changed:,} securities change BM group when the comparison becomes conditional on size.')
fig, axes = plt.subplots(1,2,figsize=(10,4.3),layout='constrained')
maximum = max(table.to_numpy().max() for table in counts.values())
for ax,(name,table) in zip(axes,counts.items()):
    im=ax.imshow(table.to_numpy(),cmap='Blues',vmin=0,vmax=maximum,aspect='auto')
    for (row,col),value in np.ndenumerate(table.to_numpy()):
        ax.text(col,row,str(value),ha='center',va='center',color='white' if value>maximum*.55 else NAVY)
    ax.set(xticks=range(5),xticklabels=range(1,6),yticks=range(5),yticklabels=range(1,6),
        xlabel='BM group (low → high)',ylabel='Size group (small → big)',title=name)
fig.colorbar(im,ax=axes,label='Number of eligible security records',shrink=.8)
fig.suptitle('June 2020: common versus size-conditional BM boundaries',fontsize=13)
plt.show()


*Empirical membership counts, June 2020.* Both panels contain exactly the same eligible securities. In the saved comparison, 1,631 of 2,925 securities change BM group, despite the size assignments and eligible universe remaining fixed. Differences arise from changing the reference comparison for BM, rather than from changing prices or subsequent returns. NYSE reference quantiles do not guarantee equal counts across the broader exchange universe.

In the independent design, a given BM boundary has the same numerical meaning across size groups. In the conditional design, each size group supplies its own reference distribution. More evenly populated cells, where achieved, do not establish matching continuous characteristics or neutral factor exposures.


### 2.3 Historical characteristic-portfolio returns

The June cross-section illustrates how sorting rules change portfolio membership. For the historical comparison, I construct $5\times5$ portfolios across size × BM, size × profitability and size × investment. The existing annual-assignment and portfolio-return functions use each June cross-section, NYSE lower-interpolation quintiles, July–June labels, and market equity from exactly the preceding calendar month. A missing calendar month invalidates its lagged weight. December t−1 presence is checked for June t, June holding-year flags from the old replication are not used as current formation-history flags.

The figure below retains the existing arithmetic calculation: arithmetic mean monthly **total returns**, with size increasing down the rows and the other characteristic increasing across columns. The common scale supports comparison across the grids. Each cell uses its available observations under the stated local preparation convention. Coverage is reported separately for each family, so the grids do not silently impose a new common sample.


In [ ]:
from factor_asset_pricing.construction import characteristic_panel, annual_test_portfolios
from factor_asset_pricing.french_data import load_french
from factor_asset_pricing.validation import validate_portfolios
# Build the full history using the independent annual source table.
panel_history, annual_history = load_prepared_data(PROCESSED, RAW_WRDS)
measured_history = characteristic_panel(panel_history, annual_history)
portfolio_frames = []
for family, characteristic in [("size_bm", "bm"), ("size_op", "op"), ("size_inv", "inv")]:
    returns, breaks = annual_test_portfolios(measured_history, characteristic)
    returns["family"] = family
    portfolio_frames.append(returns)
portfolios = pd.concat(portfolio_frames, ignore_index=True)
assert not portfolios.duplicated(["family", "mdate", "row_portfolio", "column_portfolio"]).any()
assert portfolios.ret.dropna().ge(-1).all()


In [ ]:
for family in ['size_bm','size_op','size_inv']:
    sample = portfolios.loc[portfolios.family.eq(family)]
    observed = sample.loc[sample.ret.notna()]
    counts_by_cell = observed.groupby(['row_portfolio','column_portfolio']).size()
    print(f'{family}: {observed.mdate.min():%Y-%m}–{observed.mdate.max():%Y-%m}, '
          f'{observed.mdate.nunique()} observed months, '
          f'{counts_by_cell.min()}–{counts_by_cell.max()} observations per cell.')


In [ ]:
means={}
for family,label in [("size_bm","Book-to-market"),("size_op","Profitability"),("size_inv","Investment")]:
    data=portfolios.loc[portfolios.family.eq(family)]
    means[label]=data.pivot_table(index="row_portfolio",columns="column_portfolio",values="ret",aggfunc="mean").astype(float)*100
fig,axes=plt.subplots(1,3,figsize=(12.5,4),constrained_layout=True)
lower=min(t.min().min() for t in means.values())
upper=max(t.max().max() for t in means.values())
for ax,(label,table) in zip(axes,means.items()):
    image=ax.imshow(table,cmap="RdBu_r",aspect="auto",vmin=lower,vmax=upper)
    for (r,c),value in np.ndenumerate(table.to_numpy()): ax.text(c,r,f"{value:.2f}",ha="center",va="center",fontsize=7)
    ax.set(title=f"Size × {label}",xlabel="Characteristic portfolio (low → high)",ylabel="Size portfolio (small → big)",xticks=range(5),xticklabels=range(1,6),yticks=range(5),yticklabels=range(1,6))
fig.colorbar(image,ax=axes,shrink=.75,label="Average monthly return (%)")
fig.suptitle("Average returns across characteristic portfolios",weight="bold")
plt.show()


The historical BM means show a pronounced low-to-high difference among the smallest portfolios, from roughly 0.73% to 1.35% per month, while the largest-size row has only a small positive endpoint difference, about 0.99% to 1.01%. High-profitability portfolios have higher reported mean returns than low-profitability portfolios in each size row. Lower-investment portfolios also have higher reported means than the highest-investment portfolios, although the interior cells are not uniformly monotonic. These observed patterns motivate characteristic comparisons without supplying a single economic explanation for them. Reading across a row compares characteristic portfolios within a size category. Reading down a column examines how that comparison varies with firm size.

These are total-return means, not alphas. Levels include the risk-free component and can reflect different benchmark exposures. Subtracting a matched RF changes portfolio levels, while a long–short comparison on the same dates cancels the common RF. Differences between family coverage also limit direct comparisons across grids.

The application has compared assignments from dated characteristics and examined the existing historical portfolio returns. [03](03_factor_construction.ipynb) takes selected portfolio contrasts and specifies the long and short positions that make them traded factors. [04](04_time_series_factor_regressions.ipynb) asks what exposures and abnormal mean remain relative to an existing benchmark.


### 2.4 Comparison with published portfolios

As a check, I compare each constructed portfolio with its published value-weighted French counterpart over matching months. Correlation, mean differences and Newey–West regression diagnostics measure agreement. Exact equality is not imposed. The local portfolios can differ from French definitions because of the linked security universe, accounting expense rule, source vintages and simple lagged-capitalization weights. The original replication comparison is recorded separately in the migration notes, and is not a notebook dependency.


In [ ]:
validation_rows = []
for family in ["size_bm", "size_op", "size_inv"]:
    published = load_french(family, ROOT / "data/raw/french").rename_axis("mdate").reset_index().melt(id_vars="mdate", var_name="asset", value_name="ret")
    published[["row_portfolio", "column_portfolio"]] = published.asset.str.extract(r"S(\d)C(\d)").astype(int)
    comparison = validate_portfolios(portfolios.loc[portfolios.family.eq(family)], published,
        portfolio_cols=["row_portfolio", "column_portfolio"], date_col="mdate", hac_lags=12)
    comparison["family"] = family
    validation_rows.append(comparison)
portfolio_validation = pd.concat(validation_rows, ignore_index=True)
display(portfolio_validation.groupby("family").agg(
    portfolios=("correlation", "size"), median_correlation=("correlation", "median"),
    min_correlation=("correlation", "min"), mean_return_difference=("mean_difference", "mean")))


## References

- Fama, E. F., and French, K. R. (2015). A five-factor asset pricing model. *Journal of Financial Economics*, 116(1), 1–22. [DOI](https://doi.org/10.1016/j.jfineco.2014.10.010).
- Kenneth French, [five-factor construction documentation](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html).
